In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# Import PyTorch and inspect the compute device available to the runtime.
import torch

# Determine whether CUDA is available for GPU computation.
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Select the GPU when available; otherwise, use the CPU.
device = 0 if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU. Enable a Colab GPU for faster training.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
%pip install -q ultralytics

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import YOLO and verify that the package is available.
from ultralytics import YOLO

print("Ultralytics YOLO is ready.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics YOLO is ready.


In [5]:
import gdown

# Google Drive file link provided by the user
shareable_link = 'https://drive.google.com/file/d/1V66gqcgcZfM4Sg3Squ6-sOwQ3jVSiJaX/view?usp=sharing'

# Extract the file ID and download using gdown
file_id = shareable_link.split('/d/')[1].split('/')[0]
download_url = f'https://drive.google.com/uc?id={file_id}'

# Set the path where the downloaded zip should be saved
zip_path = 'ppe_dataset.zip'

gdown.download(download_url, zip_path, quiet=False)
print("Downloaded dataset archive to:", zip_path)

Downloading...
From (original): https://drive.google.com/uc?id=1V66gqcgcZfM4Sg3Squ6-sOwQ3jVSiJaX
From (redirected): https://drive.google.com/uc?id=1V66gqcgcZfM4Sg3Squ6-sOwQ3jVSiJaX&confirm=t&uuid=27e6e2c4-ccb4-42ca-a1b9-0f1b885f9164
To: /kaggle/working/ppe_dataset.zip
100%|██████████| 614M/614M [00:14<00:00, 43.1MB/s] 

Downloaded dataset archive to: ppe_dataset.zip


In [6]:
# Extract the dataset archive into Colab's working directory.
from pathlib import Path
import zipfile

extract_dir = Path("/content/ppe_data")

# Unpack the dataset files.
with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_dir)

# Find the dataset configuration file.
yaml_files = list(extract_dir.rglob("data.yaml"))

# Stop if the expected configuration file is missing.
if not yaml_files:
    raise FileNotFoundError(
        "No data.yaml found. Check the structure of your dataset ZIP."
    )

data_yaml = yaml_files[0]
print("Dataset configuration:", data_yaml)

Dataset configuration: /content/ppe_data/Dataset_no_person_glasses/data.yaml


In [7]:
# Inspect the extracted folder structure to verify train/val/test subfolder names
import os
dataset_root = "/content/ppe_data/Dataset_no_person_glasses"
if os.path.exists(dataset_root):
    print("Subdirectories inside dataset root:", os.listdir(dataset_root))
    for sub in os.listdir(dataset_root):
        sub_path = os.path.join(dataset_root, sub)
        if os.path.isdir(sub_path):
            print(f"  {sub}/ contents:", os.listdir(sub_path)[:5])
else:
    print(f"Directory {dataset_root} does not exist. Checked /content/ppe_data:", os.listdir("/content/ppe_data"))

Subdirectories inside dataset root: ['test', 'data.yaml', 'train', 'val']
  test/ contents: ['images', 'labels']
  train/ contents: ['images', 'labels']
  val/ contents: ['images', 'labels']


In [8]:
# Load pretrained YOLOv8 nano weights for transfer learning.
model = YOLO("yolov8n.pt")

# Display the model's class names before custom training.
print(model.names)

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [9]:
%pip install --force-reinstall sympy==1.12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 35.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.8/567.8 kB 13.2 MB/s  0:00:00
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy━━━━━━━━━━━━━━━━━━━ 0/2 [mpmath]
    Found existing installation: sympy 1.14.0 0/2 [mpmath]
    Uninstalling sympy-1.14.0:0m╺━━━━━━━━━━━━━━━━━━━ 1/2 [sympy]
      Successfully uninstalled sympy-1.14.0━━━━━━━━━━━━━━━━━━━ 1/2 [sympy]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sympy]32m1/2 [sympy]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires sympy>=1.13.3, but you have sympy 1.12 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import yaml

# Read the existing data.yaml configuration
with open(data_yaml, 'r') as f:
    yaml_data = yaml.safe_load(f)

# Programmatically rewrite the dataset root and correct relative paths for Google Colab
yaml_data['path'] = '/content/ppe_data/Dataset_no_person_glasses'
yaml_data['train'] = 'train/images'
yaml_data['val'] = 'val/images'
if 'test' in yaml_data:
    yaml_data['test'] = 'test/images'

# Save the corrected configuration file
with open(data_yaml, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("Successfully corrected paths in data.yaml!")
print(yaml_data)

Successfully corrected paths in data.yaml!
{'path': '/content/ppe_data/Dataset_no_person_glasses', 'train': 'train/images', 'val': 'val/images', 'test': 'test/images', 'nc': 5, 'names': ['Helmet', 'Mask', 'Safety Vest', 'boots', 'glove']}


In [13]:
# 1. Fine-tune YOLOv8 on your custom dataset (Kaggle paths)
from pathlib import Path
import shutil
from IPython.display import FileLink, display

results = model.train(
    data=str(data_yaml),
    epochs=30,
    imgsz=640,
    batch=16,
    device=device,
    workers=2,
    patience=5,
    deterministic=False,

    # --- weight regularization ---
    weight_decay=0.001,
    freeze=10,

    # --- learning rate schedule ---
    lr0=0.005,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,

    # --- data augmentation ---
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=5,
    momentum=0.9,

    project="/kaggle/working/runs",   # Kaggle: only /kaggle/working is saved and downloadable
    name="ppe_yolov8n",
    exist_ok=True,
)

# 2. Locate and load the best checkpoint (use save_dir so the path is always right)
run_dir = Path(results.save_dir)
best_weights = run_dir / "weights" / "best.pt"
print("Run folder:", run_dir)

if best_weights.exists():
    best_model = YOLO(str(best_weights))
    print("Best checkpoint loaded successfully!")
else:
    raise FileNotFoundError(f"best.pt was not found in {run_dir / 'weights'}")

# 3. Evaluate the best checkpoint on the held-out test split
test_metrics = best_model.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    device=device,
    project="/kaggle/working/runs",
    name="ppe_yolov8n_test",
    exist_ok=True,
)

# 4. Print the main box-detection metrics
print("mAP50:", test_metrics.box.map50)
print("mAP50-95:", test_metrics.box.map)

# 5. Copy best.pt to /kaggle/working and show a download link
shutil.copy(best_weights, "/kaggle/working/best.pt")
display(FileLink("best.pt"))                      # click to download

Ultralytics 8.4.192 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/ppe_data/Dataset_no_person_glasses/data.yaml, degrees=10.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.9, mosaic=1.0, multi_scale=0.0, name=ppe_yolov8n, nbs=64, nms=None, opset=No

/kaggle/working/best.pt